# 核心架构：HybridFlow 混合编程模型

##### verl 的核心创新是 HybridFlow 架构，它采用<font color='red'>分层混合控制策略</font>：

| 层级                  | 控制模式                        | 职责                                                            |
| ------------------- | --------------------------- | ------------------------------------------------------------- |
| **节点间（Inter-Node）** | **<font color='blue'>单控制器（Single Controller）</font>** | <font color='red'>由 `RayPPOTrainer` 统一编排不同模型（Actor、Critic、Ref、Reward）的执行顺序和数据依赖</font> |
| **节点内（Intra-Node）** | **<font color='blue'>多控制器（Multi Controller）</font>**  | 每个模型内部使用 Megatron-LM 或 vLLM 的原生分布式并行，各自管理 TP/PP/DP            |

这种设计的优势：单控制器保证算法灵活性，多控制器保证分布式计算效率

# 三大组件的分工

In [ ]:
┌─────────────────────────────────────────────────────────────────┐
│                     Single Controller (Ray Driver)              │
│                      RayPPOTrainer.fit()                        │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐             │
│  │  加载数据    │→│  Rollout    │→│  训练更新    │             │
│  │  (Prompts)  │  │  (生成响应)  │  │  (PPO/GRPO) │             │
│  └─────────────┘  └─────────────┘  └─────────────┘             │
└─────────────────────────────────────────────────────────────────┘
           ↓ Ray RPC 调用                    ↓ Ray RPC 调用
┌─────────────────────────────────────────────────────────────────┐
│                    Worker Groups (Ray Actors)                   │
│  ┌──────────────────┐  ┌──────────────────┐                     │
│  │ ActorRolloutRef  │  │   CriticWorker   │                     │
│  │   WorkerGroup    │  │    WorkerGroup   │                     │
│  │  (Megatron + vLLM)│  │   (Megatron)    │                     │
│  └──────────────────┘  └──────────────────┘                     │
└─────────────────────────────────────────────────────────────────┘
           ↓ 内部多控制器并行
┌─────────────────────────────────────────────────────────────────┐
│              Megatron-LM (训练)  │  vLLM (推理生成)              │
│  ┌─────────────────────────────┐  ┌─────────────────────────┐ │
│  │  TP + PP + DP 3D并行         │  │  TP + DP 并行 (SPMD)    │ │
│  │  前向/反向/梯度更新           │  │  自回归生成               │ │
│  │  AllGather/ReduceScatter      │  │  KV Cache 管理           │ │
│  └─────────────────────────────┘  └─────────────────────────┘ │
└─────────────────────────────────────────────────────────────────┘

# 关键组件详解
## 1. Ray：分布式编排层
Ray 在 verl 中承担资源管理与任务调度的角色：
- RayPPOTrainer：单控制器，运行在 driver 进程，用 Python 写标准单进程算法逻辑
- RayWorkerGroup：将一组 Ray Actor 封装为 Worker Group，对 controller 暴露统一 API
- ResourcePoolManager：管理 GPU 资源的分配和放置策略 

##### 关键抽象：@register(dispatch_mode=...) 装饰器

In [ ]:
# 在 Worker 上注册方法，声明数据分发模式
@register(dispatch_mode=Dispatch.ONE_TO_ALL)
def init_model(self):
    # 所有 worker 都执行模型初始化
    ...

@register(dispatch_mode=Dispatch.MEGATRON_PP_AS_DP_PROTO)
def generate_sequences(self, prompts: DataProto):
    # PP 维度被视为 DP 维度进行数据分发
    ...

@register(dispatch_mode=Dispatch.MEGATRON_COMPUTE_PROTO)
def update_actor(self, data: DataProto):
    # 数据按 DP 维度分发，TP/PP 组内全广播
    ...

##### Controller 调用这些方法时，Ray 自动完成：
- 数据分发（dispatch）：将 DataProto 拆分到各 worker
- 并行执行：各 worker 在本地 GPU 上执行计算
- 结果收集（collect）：按约定模式聚合结果返回 controller 

## Megatron-LM：训练后端
##### verl 的 Megatron 后端支持 5D 并行（TP + EP + CP + DP + PP）：

| Worker 类型               | 职责                             | 对应 Megatron 组件                     |
| ----------------------- | ------------------------------ | ---------------------------------- |
| `ActorRolloutRefWorker` | Actor 训练 + Rollout 生成 + Ref 模型 | `MegatronPPOActor` + `vLLMRollout` |
| `CriticWorker`          | Critic 模型<font color='red'>训练 </font>                   | `MegatronPPOCritic`                |
| `RewardWorker`          | Reward 模型<font color='red'>推理 </font>                   | `MegatronRewardModel`              |

##### 关键类：
- MegatronWorker：基类，提供 get_megatron_global_info() 和 get_megatron_rank_info() 获取 3D 并行信息
- MegatronPPOActor：封装 PPO 训练逻辑（计算 log prob、梯度更新）
- AllGatherPPModel：在 PP 维度上聚合模型权重，支持重分片 

## vLLM：推理生成后端

##### <font color='red'>verl 修改了 vLLM 引擎，使其以 SPMD（单程序多数据） 模式运行，适配 WorkerGroup 设计：</font>

In [ ]:
# vLLM Rollout 初始化（在 Worker 内部）
rollout = vLLMRollout(
    actor_module=params,           # 从 Megatron 获取的模型参数
    config=self.config.rollout,
    tokenizer=self.tokenizer,
    model_hf_config=self.actor_model_config,
    train_tp=mpu.get_tensor_model_parallel_world_size()  # 训练时的 TP 大小
)

### 关键修改：
- <font color='red'>替换 vLLM 的集中式 KV Cache 管理器为分布式管理器，与多控制器范式对齐</font>
- 支持 per-tensor weight loading（vLLM ≥ 0.8.2），实现细粒度权重更新 


# 3D-HybridEngine：Megatron 与 vLLM 的桥梁

##### 这是 verl 最核心的技术创新，<font color='red'>解决 Actor 模型在训练（Megatron）和生成（vLLM）之间的权重同步问题。</font>

## 问题背景
- 训练阶段：Actor 使用 Megatron 的 3D 并行（如 TP=8, PP=4, DP=2）
- 生成阶段：vLLM 使用不同的并行策略（如 TP=4, DP=16）以获得更高吞吐
- 传统方案：需要拷贝全量权重，通信开销巨大（70B 模型每次切换需传输 140GB）

## verl 的解决方案

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    3D-HybridEngine                           │
│                                                              │
│   ┌──────────────┐         ┌──────────────┐                 │
│   │  Megatron    │         │    vLLM      │                 │
│   │  Actor Model │←───────→│  Rollout     │                 │
│   │  (训练布局)   │ 重分片   │  (生成布局)   │                 │
│   │  TP×PP×DP    │         │  TP×DP       │                 │
│   └──────────────┘         └──────────────┘                 │
│          ↑                        ↑                         │
│   ┌─────────────────────────────────────────┐               │
│   │    MegatronVLLMShardingManager            │               │
│   │    (Context Manager)                     │               │
│   │  - 在 GPU 内存中维护双缓冲               │               │
│   │  - 使用 NCCL AllGather 在 PP/TP 维度聚合  │               │
│   │  - 原地重分片，零内存冗余                  │               │
│   └─────────────────────────────────────────┘               │
└─────────────────────────────────────────────────────────────┘

### 核心实现（约 2.4k 行代码）：
- 双缓冲内存管理：
  - <font color='red'>训练权重和生成权重分别存储在独立内存缓冲区</font>
  - 训练时生成权重 offload 到 CPU，生成时重新加载到 GPU
- NCCL 重分片：
  - 在 micro DP 组内使用 all_gather 收集和拼接参数
  - 在 TP 维度内通过 MegatronVLLMShardingManager 处理数据重分片
- 零内存冗余：
  - <font color='red'>不维护两份完整模型副本，通过原地转换实现布局切换 </font>

# 完整调用流程（以 PPO 为例）

In [ ]:
Step 1: 初始化 (Ray Driver)
─────────────────────────
RayPPOTrainer
  └── ResourcePoolManager 分配 GPU
      ├── ActorRolloutRefWorkerGroup → GPU 0-31 (TP=8, PP=4)
      ├── CriticWorkerGroup → GPU 32-47
      └── RewardWorkerGroup → GPU 48-63

Step 2: Worker 初始化 (Ray RPC)
──────────────────────────────
actor_rollout_ref_wg.init_model()  # @ONE_TO_ALL
  ├── 每个 Worker 上:
  │   ├── Megatron 初始化 Actor (TP×PP×DP)
  │   ├── vLLM 初始化 Rollout (修改版 SPMD 引擎)
  │   └── MegatronVLLMShardingManager 注册重分片映射
  └── 返回: 各 Worker 的 3D 并行信息

Step 3: Rollout 生成
────────────────────
actor_rollout_ref_wg.generate_sequences(prompts)  # @MEGATRON_PP_AS_DP_PROTO
  ├── Controller dispatch: 将 prompts 按 Rollout 的 DP 维度分发
  ├── 每个 Worker:
  │   ├── ShardingManager 将 Actor 权重重分片为 vLLM 布局
  │   ├── vLLM 执行自回归生成
  │   ├── Actor 计算 old_log_prob
  │   └── ShardingManager 恢复 Actor 训练布局
  └── Controller collect: 聚合生成的 sequences + log_probs

Step 4: 准备阶段 (Critic/Ref/Reward 前向)
─────────────────────────────────────────
critic_wg.compute_values(data)      # @MEGATRON_COMPUTE_PROTO
ref_wg.compute_ref_log_prob(data)   # @MEGATRON_COMPUTE_PROTO
reward_wg.compute_rm_score(data)   # @MEGATRON_COMPUTE_PROTO

Step 5: 训练更新
────────────────
actor_rollout_ref_wg.update_actor(data)  # @MEGATRON_COMPUTE_PROTO
  ├── 数据按 DP 分发到各 TP/PP 组
  ├── Megatron 执行 PPO 前向 + 反向
  └── 只从 tp=0 和 last pp stage 收集输出

critic_wg.update_critic(data)  # 同理

# 两种 Rollout 模式
verl 支持两种生成模式，适应不同场景：

| 模式               | 名称         | 特点                                          | 适用场景               |
| ---------------- | ---------- | ------------------------------------------- | ------------------ |
| **HybridEngine** | 共置/SPMD 模式 | Actor 和 Rollout 在同一 WorkerGroup，通过内存重分片共享权重 | 标准 RLHF，无工具调用      |
| **AsyncServer**  | 异步服务器模式    | vLLM/SGLang 作为独立推理服务器，通过 RPC 调用             | 多轮对话、工具调用、Agent 训练 |

### AsyncServer 模式下：
- vLLM/SGLang 作为独立进程运行
- 权重更新通过 load_weights API 或 CheckpointEngine 完成
- AgentLoop 管理对话状态机，支持 out-of-order 返回 

# 总结
verl 通过 HybridFlow 架构实现了三者的无缝组合：

| 组件              | 职责           | verl 中的角色                             |
| --------------- | ------------ | ------------------------------------- |
| **Ray**         | 分布式资源管理与任务编排 | 单控制器（`RayPPOTrainer`）+ WorkerGroup 抽象 |
| **Megatron-LM** | 大模型分布式训练     | 训练后端（3D/5D 并行 + PPO 计算逻辑）             |
| **vLLM**        | 高吞吐推理生成      | 推理后端（修改版 SPMD 引擎）                     |

### 核心价值：
- 解耦：算法逻辑（单进程 Python）与分布式执行（Ray + Megatron/vLLM）完全分离
- 高效：3D-HybridEngine 实现零冗余、低通信的权重重分片
- 灵活：同一套代码支持 FSDP/Megatron + vLLM/SGLang 多种后端组合 
